# Assignment 3: k-Nearest Neighbour (kNN)

**Objective:** Implement k-Nearest Neighbour algorithm using Python ML libraries and analyze predictions.

**Dataset Used:** Breast Cancer (Diagnostic)
---

## Q1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully.')

Libraries imported successfully.


## Q2: Load Breast Cancer Dataset

In [ ]:
# Load dataset from UCI ML Repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data'

# Define column names: ID, Diagnosis, then 30 feature columns
base_features = ['radius','texture','perimeter','area','smoothness',
                 'compactness','concavity','concave_points','symmetry','fractal_dimension']
columns = ['id', 'diagnosis'] + \
          [f'{f}_mean'   for f in base_features] + \
          [f'{f}_se'     for f in base_features] + \
          [f'{f}_worst'  for f in base_features]

df = pd.read_csv(url, names=columns)
print('Dataset loaded successfully!')
df.head()

Dataset loaded successfully!


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## Q3: Explore the Dataset

In [ ]:
print('Shape (rows, columns):', df.shape)
print('\nDiagnosis Classes:', df['diagnosis'].unique())
print('\nClass Distribution:')
print(df['diagnosis'].value_counts())
print('\nMissing Values:', df.isnull().sum().sum())

Shape (rows, columns): (569, 32)

Diagnosis Classes: ['M' 'B']

Class Distribution:
diagnosis
B    357
M    212
Name: count, dtype: int64

Missing Values: 0


In [ ]:
# Statistical summary of first 10 feature columns
df.iloc[:, 2:12].describe().round(3)

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean
count,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000
mean,14.127,19.290,91.969,654.889,0.096,0.104,0.089,0.049,0.181,0.063
std,3.524,4.301,24.299,351.914,0.014,0.053,0.080,0.039,0.027,0.007
min,6.981,9.710,43.790,143.500,0.053,0.019,0.000,0.000,0.106,0.050
25%,11.700,16.170,75.170,420.300,0.086,0.065,0.030,0.020,0.162,0.058
50%,13.370,18.840,86.240,551.100,0.096,0.093,0.062,0.034,0.179,0.062
75%,15.780,21.800,104.100,782.700,0.105,0.130,0.131,0.074,0.196,0.066
max,28.110,39.280,188.500,2501.000,0.163,0.345,0.427,0.201,0.304,0.097


## Q4: Split Features (X) and Target (y)

In [ ]:
# Drop ID column — not a useful feature
df.drop(columns=['id'], inplace=True)

# Encode target: M (Malignant) = 1, B (Benign) = 0
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])   # B=0, M=1

# Features and Target
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

print('Feature matrix shape (X):', X.shape)
print('Target vector shape  (y):', y.shape)
print('\nEncoding:', dict(zip(le.classes_, le.transform(le.classes_))))
X.head()

Feature matrix shape (X): (569, 30)
Target vector shape  (y): (569,)

Encoding: {'B': np.int64(0), 'M': np.int64(1)}


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## Q5: Feature Scaling (Essential for kNN)

In [ ]:
# kNN is distance-based — unscaled features bias the distance calculation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print('Scaling applied. Sample means (should be ~0):')
print(X_scaled.mean().round(4).head(5))

Scaling applied. Sample means (should be ~0):
radius_mean       -0.0
texture_mean       0.0
perimeter_mean    -0.0
area_mean         -0.0
smoothness_mean   -0.0
dtype: float64


## Q6: Train-Test Split (70:30)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

print('Training set size:', X_train.shape)
print('Testing  set size:', X_test.shape)
print('\nClass distribution in Training set:')
print(y_train.value_counts().rename({0: 'Benign', 1: 'Malignant'}))
print('\nClass distribution in Test set:')
print(y_test.value_counts().rename({0: 'Benign', 1: 'Malignant'}))

Training set size: (398, 30)
Testing  set size: (171, 30)

Class distribution in Training set:
diagnosis
Benign       250
Malignant    148
Name: count, dtype: int64

Class distribution in Test set:
diagnosis
Benign       107
Malignant     64
Name: count, dtype: int64


## Q7: Build kNN Model (k = 3)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
knn.fit(X_train, y_train)
print('kNN model trained with k =', knn.n_neighbors)
print('Distance metric used:', knn.metric)

kNN model trained with k = 3
Distance metric used: euclidean


## Q8: Make Predictions

In [ ]:
y_pred = knn.predict(X_test)
y_pred_proba = knn.predict_proba(X_test)

print('First 10 predicted labels:', list(y_pred[:10]))
print('Decoded:', ['Malignant' if p == 1 else 'Benign' for p in y_pred[:10]])

First 10 predicted labels: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0)]
Decoded: ['Benign', 'Benign', 'Benign', 'Benign', 'Benign', 'Benign', 'Malignant', 'Benign', 'Malignant', 'Benign']


## Q9: Show Correct and Wrong Predictions

In [ ]:
results = pd.DataFrame({
    'Actual':    y_test.values,
    'Predicted': y_pred
})
results['Actual_Label']    = results['Actual'].map({0: 'Benign', 1: 'Malignant'})
results['Predicted_Label'] = results['Predicted'].map({0: 'Benign', 1: 'Malignant'})
results['Status']          = results.apply(
    lambda r: 'Correct' if r.Actual == r.Predicted else 'Wrong', axis=1
)

print('Summary:')
print(results['Status'].value_counts())
print(f'\nWrong predictions: {(results["Status"] == "Wrong").sum()} out of {len(results)}')

results[['Actual_Label', 'Predicted_Label', 'Status']].head(20)

Summary:
Status
Correct    164
Wrong        7
Name: count, dtype: int64

Wrong predictions: 7 out of 171


,Actual_Label,Predicted_Label,Status
0,Benign,Benign,Correct
1,Benign,Benign,Correct
2,Benign,Benign,Correct
3,Benign,Benign,Correct
4,Benign,Benign,Correct
5,Benign,Benign,Correct
6,Malignant,Malignant,Correct
7,Benign,Benign,Correct
8,Malignant,Malignant,Correct
9,Benign,Benign,Correct


In [ ]:
# Show only misclassified records
wrong = results[results['Status'] == 'Wrong']
print('Misclassified Records:')
wrong[['Actual_Label', 'Predicted_Label', 'Status']]

Misclassified Records:


,Actual_Label,Predicted_Label,Status
21,Malignant,Benign,Wrong
65,Malignant,Benign,Wrong
111,Malignant,Benign,Wrong
125,Malignant,Benign,Wrong
148,Malignant,Benign,Wrong
159,Malignant,Benign,Wrong
168,Malignant,Benign,Wrong


## Q10: Evaluate Model Performance

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc * 100:.2f}%')

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
print(cm)
print('  (Rows = Actual, Columns = Predicted)')
print('  Labels: [Benign=0, Malignant=1]')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

Accuracy: 95.91%

Confusion Matrix:
[[107   0]
 [  7  57]]
  (Rows = Actual, Columns = Predicted)
  Labels: [Benign=0, Malignant=1]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.94      1.00      0.97       107
   Malignant       1.00      0.89      0.94        64

    accuracy                           0.96       171
   macro avg       0.97      0.95      0.96       171
weighted avg       0.96      0.96      0.96       171



## Q11: Find the Best Value of k (Elbow Method)

In [ ]:
# Test k from 1 to 20 and track accuracy
k_values  = range(1, 21)
train_acc = []
test_acc  = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    model.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, model.predict(X_train)))
    test_acc.append(accuracy_score(y_test, model.predict(X_test)))

# Display as table
k_table = pd.DataFrame({'k': list(k_values), 'Train Accuracy': train_acc, 'Test Accuracy': test_acc})
k_table['Train Accuracy'] = k_table['Train Accuracy'].round(4)
k_table['Test Accuracy']  = k_table['Test Accuracy'].round(4)

best_k = k_table.loc[k_table['Test Accuracy'].idxmax(), 'k']
print(f'Best k = {best_k} with Test Accuracy = {k_table["Test Accuracy"].max():.4f}')
k_table

Best k = 5 with Test Accuracy = 0.9649


,k,Train Accuracy,Test Accuracy
0,1,1.0000,0.9415
1,2,0.9648,0.9357
2,3,0.9849,0.9591
3,4,0.9724,0.9591
4,5,0.9724,0.9649
5,6,0.9598,0.9591
6,7,0.9698,0.9649
7,8,0.9648,0.9532
8,9,0.9698,0.9532
9,10,0.9648,0.9532


## Q12: Retrain with Best k & Final Evaluation

In [ ]:
best_knn = KNeighborsClassifier(n_neighbors=int(best_k), metric='euclidean')
best_knn.fit(X_train, y_train)
y_pred_best = best_knn.predict(X_test)

print(f'Final kNN Model — k = {best_k}')
print(f'Accuracy: {accuracy_score(y_test, y_pred_best) * 100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Benign', 'Malignant']))

Final kNN Model — k = 5
Accuracy: 96.49%

Classification Report:
              precision    recall  f1-score   support

      Benign       0.95      1.00      0.97       107
   Malignant       1.00      0.91      0.95        64

    accuracy                           0.96       171
   macro avg       0.97      0.95      0.96       171
weighted avg       0.97      0.96      0.96       171



## Conclusion

> **Dataset:** Breast Cancer Wisconsin (Diagnostic) — 569 samples, 30 features
>
> **Algorithm:** kNN classifies a new sample by looking at its **k nearest neighbours** in the feature space and voting on the majority class.
>
> **Key Findings:**
> - With **k = 3**, the model achieves good accuracy on the test set.
> - The elbow method helps identify the **optimal k** that balances bias and variance too small a k overfits, too large a k underfits.
> - **Feature scaling was critical** here without it, large-scale features like `area_mean` would distort distances.
> - Most misclassifications tend to occur near the **decision boundary** where Benign and Malignant samples overlap in feature space.
> - **Precision for Malignant** is the most important metric in medical diagnosis we want to minimise false negatives (predicting Benign when actually Malignant).

---
